Compare four protocol sources using expert criterion ratings, paired case tests, Holm correction and bootstrap inte.


In [1]:
from pathlib import Path
import hashlib, itertools, json
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from pathlib import Path
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'evaluations/human_and_model_validation/blind_protocols_60.csv').is_file() and (p / 'data').is_dir()))
BASE = ROOT / 'evaluations/human_and_model_validation'
OUT = BASE / 'expert_agreement'
OUT.mkdir(parents=True, exist_ok=True)
H = ['expert_01', 'expert_02', 'expert_03', 'expert_04']
S = ['delphi', 'deepseekv4', 'gpt', 'qwen']
B = 10000
SEED = 20261004

def bounds(x):
    return np.quantile(x, [0.025, 0.975], axis=0)

def summarize(p, rng):
    p = p[S].dropna()
    n = len(p)
    a = p.to_numpy()
    w = rng.multinomial(n, np.full(n, 1 / n), size=B)
    boot = w @ a / n
    low, high = bounds(boot)
    means = []
    for j, s in enumerate(S):
        means.append(dict(source=s, n_cases=n, mean=float(a[:, j].mean()), sd_case=float(a[:, j].std(ddof=1)), ci_low=float(low[j]), ci_high=float(high[j])))
    comparisons = []
    for i, j in itertools.combinations(range(4), 2):
        diff = a[:, i] - a[:, j]
        test = stats.ttest_rel(a[:, i], a[:, j])
        ci = bounds(boot[:, i] - boot[:, j])
        comparisons.append(dict(a=S[i], b=S[j], n_cases=n, difference=float(diff.mean()), ci_low=float(ci[0]), ci_high=float(ci[1]), t=float(test.statistic), df=n - 1, p_raw=float(test.pvalue), wins_a=int((diff > 1e-09).sum()), ties=int((np.abs(diff) <= 1e-09).sum()), wins_b=int((diff < -1e-09).sum())))
    fr = stats.friedmanchisquare(*[a[:, i] for i in range(4)])
    return (means, comparisons, dict(statistic=float(fr.statistic), p=float(fr.pvalue), n_cases=n))


In [2]:
def main():
    rng = np.random.default_rng(SEED)
    previous = json.loads((OUT / 'results.json').read_text(encoding='utf-8'))
    for name, sha in previous['input_sha256'].items():
        assert hashlib.sha256((BASE / name).read_bytes()).hexdigest() == sha, ('Changed source, rerun criterion extraction', name)
    d = pd.read_csv(OUT / 'aligned_ratings.csv', encoding='utf-8-sig')
    key = pd.read_csv(BASE / 'blind_protocols_60_key.csv', sep=';', encoding='utf-8-sig')
    assert len(key) == 60 and key.case_id.is_unique
    sources = key.melt(id_vars='case_id', value_vars=[f'source_of_{i}' for i in range(1, 5)], var_name='slot', value_name='source')
    sources['protocol'] = sources.slot.str[-1].astype(int)
    assert sources.groupby('case_id').source.apply(lambda x: set(x) == set(S)).all()
    d = d.merge(sources[['case_id', 'protocol', 'source']], on=['case_id', 'protocol'], validate='many_to_one')
    assert len(d) == 3408
    judge = pd.read_csv(BASE / 'gpt41_judging/judge_gpt41/blind_protocols_60_gpt41_judge.csv', sep=';', encoding='utf-8-sig')
    check = sources.merge(judge[['case_id', 'blind_slot', 'protocol_source']], left_on=['case_id', 'protocol'], right_on=['case_id', 'blind_slot'], validate='one_to_one')
    assert len(check) == 240 and check.source.eq(check.protocol_source).all()
    chk = d.assign(net=d.points * d['GPT-4.1'], maximum=d.points.clip(lower=0)).groupby(['case_id', 'protocol']).agg(net=('net', 'sum'), maximum=('maximum', 'sum'))
    chk['recomputed'] = (chk.net / chk.maximum).round(4)
    chk = chk.reset_index().merge(judge[['case_id', 'blind_slot', 'Judge_score']], left_on=['case_id', 'protocol'], right_on=['case_id', 'blind_slot'], validate='one_to_one')
    assert np.allclose(chk.recomputed, chk.Judge_score, atol=1e-12)
    long = d.melt(id_vars=['case_id', 'protocol', 'source', 'criterion', 'points', 'axis'], value_vars=H, var_name='human', value_name='met')
    blocks = long.groupby(['case_id', 'human']).met.agg(n='size', rated='count')
    complete_index = blocks.index[blocks.n.eq(blocks.rated)]
    long['complete'] = pd.MultiIndex.from_frame(long[['case_id', 'human']]).isin(complete_index)
    eligible = long[long.complete].copy()
    eligible['net'] = eligible.points * eligible.met
    eligible['earned'] = eligible.points.clip(lower=0) * eligible.met
    eligible['penalty'] = -eligible.points.clip(upper=0) * eligible.met
    eligible['maximum'] = eligible.points.clip(lower=0)
    scores = eligible.groupby(['case_id', 'human', 'source']).agg(net=('net', 'sum'), earned=('earned', 'sum'), penalty=('penalty', 'sum'), maximum=('maximum', 'sum'), n_criteria=('met', 'count')).reset_index()
    scores['score'] = 100 * scores.net / scores.maximum
    assert scores.groupby(['case_id', 'human']).source.nunique().eq(4).all()
    assert len(scores) == 4 * len(complete_index) and 0 < len(complete_index) <= 4 * len(key)
    per_case = scores.groupby(['case_id', 'source']).score.mean().unstack()[S]
    assert per_case.shape == (60, 4) and per_case.notna().all().all()
    overall, contrasts, omnibus = summarize(per_case, rng)
    for rec, p in zip(contrasts, multipletests([r['p_raw'] for r in contrasts], method='holm')[1]):
        rec['p_holm'] = float(p)
    expert_means = scores.groupby(['human', 'source']).score.agg(['mean', 'std', 'count']).reset_index().to_dict('records')
    axes = eligible.groupby(['case_id', 'human', 'source', 'axis']).agg(net=('net', 'sum'), maximum=('maximum', 'sum')).reset_index()
    axes['score'] = np.where(axes.maximum.gt(0), 100 * axes.net / axes.maximum, np.nan)
    axes_case = axes.groupby(['case_id', 'axis', 'source']).score.mean().unstack('source')[S]
    axis_means = []
    axis_contrasts = []
    axis_omnibus = []
    for axis in sorted(axes.axis.unique()):
        means, comparisons, omni = summarize(axes_case.xs(axis, level='axis'), rng)
        axis_means += [dict(axis=axis, **r) for r in means]
        axis_contrasts += [dict(axis=axis, **r) for r in comparisons]
        axis_omnibus.append(dict(axis=axis, **omni))
    for rec, p in zip(axis_contrasts, multipletests([r['p_raw'] for r in axis_contrasts], method='holm')[1]):
        rec['p_holm_36'] = float(p)
    full_cases = scores.groupby('case_id').human.nunique()
    full_cases = full_cases.index[full_cases.eq(4)]
    common_means, common_pairs, _ = summarize(per_case.loc[full_cases], rng)
    macro = scores.groupby(['human', 'source']).score.mean().groupby('source').mean().to_dict()
    partial = long[long.groupby(['case_id', 'human']).met.transform('count').gt(0)].copy()
    partial['lower_met'] = partial.met.fillna(partial.points.lt(0).astype(int))
    partial['upper_met'] = partial.met.fillna(partial.points.gt(0).astype(int))
    partial['lower_net'] = partial.points * partial.lower_met
    partial['upper_net'] = partial.points * partial.upper_met
    partial['maximum'] = partial.points.clip(lower=0)
    bd = partial.groupby(['case_id', 'human', 'source']).agg(lower=('lower_net', 'sum'), upper=('upper_net', 'sum'), maximum=('maximum', 'sum'))
    bd['lower_score'] = 100 * bd.lower / bd.maximum
    bd['upper_score'] = 100 * bd.upper / bd.maximum
    missing_bounds = bd.groupby(['case_id', 'source'])[['lower_score', 'upper_score']].mean().groupby('source').mean().reset_index().to_dict('records')
    contrib = axes.merge(scores[['case_id', 'human', 'source', 'maximum']], on=['case_id', 'human', 'source'], suffixes=('_axis', '_overall'), validate='many_to_one')
    contrib['contribution'] = 100 * contrib.net / contrib.maximum_overall
    contribution_case = contrib.groupby(['case_id', 'source', 'axis']).contribution.mean().unstack('axis').fillna(0)
    assert np.allclose(contribution_case.sum(axis=1).sort_index(), per_case.stack().sort_index())
    contributions = contribution_case.groupby('source').mean().to_dict('index')
    result = dict(method=dict(score='100 * sum(points * met) / sum(positive points), unclipped', aggregation='Complete human-case blocks for all 4 sources; mean across available humans within case, then equally across 60 cases', n_cases=60, n_human_case_blocks=len(complete_index), n_human_protocol_scores=len(scores), bootstrap_replicates=B, seed=SEED, tests='Two-sided paired t tests on case-mean scores; Holm across 6 overall / 36 axis comparisons; bootstrap CIs are pointwise'), overall=overall, pairwise=contrasts, friedman=omnibus, expert_means=expert_means, axis_means=axis_means, axis_pairwise=axis_contrasts, axis_friedman=axis_omnibus, sensitivity_common_cases=common_means, sensitivity_equal_expert_weight=macro, sensitivity_missing_bounds=missing_bounds, axis_contributions=contributions, coverage=blocks.reset_index().to_dict('records'), key_sha256=hashlib.sha256((BASE / 'blind_protocols_60_key.csv').read_bytes()).hexdigest(), aligned_sha256=hashlib.sha256((OUT / 'aligned_ratings.csv').read_bytes()).hexdigest())
    (OUT / 'protocol_comparison_results.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    for name, records in [('overall', overall), ('pairwise', contrasts), ('axis_means', axis_means), ('axis_pairwise', axis_contrasts), ('expert_means', expert_means)]:
        pd.DataFrame(records).to_csv(OUT / f'protocol_{name}.csv', index=False, encoding='utf-8-sig')
    scores.to_csv(OUT / 'protocol_human_scores.csv', index=False, encoding='utf-8-sig')
    per_case.to_csv(OUT / 'protocol_case_means.csv', encoding='utf-8-sig')
    print('OVERALL')
    print(pd.DataFrame(overall).to_string(index=False))
    print('PAIRS')
    print(pd.DataFrame(contrasts).to_string(index=False))
    print('BY_HUMAN')
    print(pd.DataFrame(expert_means).pivot(index='human', columns='source', values='mean').to_string())
    print('AXIS_MEANS')
    print(pd.DataFrame(axis_means).pivot(index='axis', columns='source', values='mean').to_string())
    print('AXIS_DELPHI_COMPARISONS')
    print(pd.DataFrame(axis_contrasts).query("a == 'delphi'")[['axis', 'b', 'n_cases', 'difference', 'ci_low', 'ci_high', 'p_raw', 'p_holm_36']].to_string(index=False))
    print('FRIEDMAN', omnibus)
    print('COMMON_CASES', common_means)
    print('EQUAL_EXPERT', macro)
    print('MISSING_BOUNDS', missing_bounds)
    print('CONTRIBUTIONS', contributions)


In [3]:
main()


OVERALL
    source  n_cases      mean   sd_case    ci_low   ci_high
    delphi       60 73.737417 11.261766 70.895880 76.638010
deepseekv4       60 61.969001 15.333329 58.043899 65.855862
       gpt       60 57.731734 17.372419 53.247613 61.967720
      qwen       60 56.123589 13.257539 52.777453 59.433823
PAIRS
         a          b  n_cases  difference    ci_low   ci_high         t  df        p_raw  wins_a  ties  wins_b       p_holm
    delphi deepseekv4       60   11.768416  8.920342 14.761124  7.877163  59 8.829968e-11      51     0       9 4.414984e-10
    delphi        gpt       60   16.005683 12.127334 20.267983  7.525426  59 3.482377e-10      53     0       7 1.392951e-09
    delphi       qwen       60   17.613828 14.435898 20.840465 10.545574  59 3.384809e-15      55     0       5 2.030885e-14
deepseekv4        gpt       60    4.237267  0.210775  8.291428  2.026271  59 4.726560e-02      34     1      25 9.453120e-02
deepseekv4       qwen       60    5.845412  2.645421  8.86685